In [113]:
from pprint import pprint
from random import randint

# Задача 2

Дана последовательность матриц A, B, C, … , Z таким образом, что с ними можно выполнить ассоциативные операции. Используя динамическое программирование, минимизируйте количество скалярных операций для нахождения их произведения.

##### Решение
Пусть `Mat(n x p) * Mat(p x m) = Mat(n x m)`. Итоговая матрица `n * m` ячеек по `p` произведений скаляров в каждом, всего: `m * p * n` скалярных операций.

Так как размеры соседних матриц по вертикали и горизонтали должны совпадать, заносим их однократно. Если перемножаем `Mat(2x3) * Mat(3x4) * Mat(4x2)`, в массив записываем `x = [2, 3, 4, 2]`

##### Решение с neerc.ifmo.ru (для проверки)

In [114]:
# Массив v[] — хранит все размеры матриц по порядку
v = [1, 2, 3, 4, 5, 6]
# dp[i][j] — ответ на отрезке [i, j) 
dp = [[-1 for _ in range(len(v))] for _ in range(len(v))]     

# l — включая в отрезок, r — исключая из отрезка
def mat_ifmo(l: int, r: int) -> int:
    global v, dp    
    if dp[l][r] == -1: # Если значение не посчитано
        if l == r - 1:
            dp[l][r] = 0
        else:
            dp[l][r] = float('inf')

            for i in range(l + 1, r):
                dp[l][r] = min(dp[l][r], v[l] * v[i] * v[r] +  mat_ifmo(l, i) + mat_ifmo(i, r))
    return dp[l][r]

print(mat_ifmo(0, len(v) - 1))

68


##### Решение по нахождению минимума
- Если результат для цепочки произведений, возвращаем его
- Если на входе 1 матрица, её не нужно умножать, возвращаем 0
- Если на входе больше 1 матрицы, 
    - Делим цепочку на 2 части всеми возможными способами
        - **Рекурсивно** считаем оптимальное произведение левой и правой части
        - Считаем результат перемножения левой матрицы на правую
        - Складываем результаты
    - Возвращается минимум из всех разбиений

##### Оценка сложности
Пусть есть `n` матриц. За время работы алгоритма мы рассматриваем цепочки из 1, 2, 3 и т.д. матриц. Всего `n*(n+1)/2` штук, для каждой цепочки есть `n` шагов по нахождению оптимального варианта разбиения. Итоговая сложность `O(n^3)`

In [115]:
def mat_min(n, m, x, cache):
    if cache[n][m] >= 0:
        return cache[n][m]
    if m - n < 2:     # 0 matrix
        cache[n][m] = 0
    else:             # n matrixes
        cache[n][m] = float('inf')
        for p in range(n + 1, m):
            L = mat_min(n, p, x, cache)     # [n x p] matrix (left)
            P = x[n] * x[p] * x[m]          # [n x p] * [p x m]
            R = mat_min(p, m, x, cache)     # [p x m] matrix (right)
            cache[n][m] = min(cache[n][m], L + P + R)
    return cache[n][m]

def mat_minimum(*x):
    cache = [[-1 for _ in range(len(x))] for _ in range(len(x))]
    return mat_min(0, len(x) - 1, x, cache)
    
print(mat_minimum(1, 2, 3, 4, 5, 6))


68


##### Нахождение минимума + порядка умножений
Одновременно с сохранением минимального числа операций сохраняем порядок оптимального варианта. Для этого берём части, полученные на предыдущем шаге и убираем лишние скобки. Базовым случаем является единственная матрица без произведений.

In [135]:
def char(x: int)->str:
    return chr(x + ord('A')) if x < 26 else chr(x + ord('a') - 26)


def mat_prod(n, m, x, cache, matrix):
    if cache[n][m] >= 0:  # result in cache
        return cache[n][m]
    if m - n < 2:  # not matrix's product
        matrix[n][m] = f'{char(n)}'
        cache[n][m] = 0
    else:
        cache[n][m] = float('inf')
        for p in range(n + 1, m):
            # (left matrix's product [n x p]) + ([n x p] * [p x m]) + ([p x m] right matrix's product)
            res = mat_prod(n, p, x, cache, matrix) + (x[n] * x[p] * x[m]) + mat_prod(p, m, x, cache, matrix)
            if res < cache[n][m]:
                cache[n][m] = res
                v = matrix[n][p]
                matrix[n][m] = f'({v[1:-1] if v[0] == '(' and v[-1] == ')' else v}*{matrix[p][m]})'
    return cache[n][m]

def mat_product(*x):
    # Init recursion
    if len(x) < 3:
        print('No products')
        return
    cache = [[-1 for _ in range(len(x))] for _ in range(len(x))]
    matrix = [['' for _ in range(len(x))] for _ in range(len(x))]
    scalars = mat_prod(0, len(x) - 1, x, cache, matrix)
    product = matrix[0][len(x) - 1]
    print(f'Min {scalars} products at {product}')
    
mat_product(1, 2, 3, 4, 5)
mat_product(10, 9, 8, 7, 7)
mat_product(6, 4, 6, 3, 8)
mat_product(13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13)
mat_product(*[randint(1, 1000) for _ in range(26)])


Min 38 products at (A*B*C*D)
Min 1526 products at (A*(B*(C*D)))
Min 288 products at (A*(B*C)*D)
Min 1623 products at (A*(B*(C*(D*(E*(F*(G*(H*(I*(J*(K*L))))))))))*(M*N*O*P*Q*R*S*T*U*V*W*X*Y))
Min 139998760 products at (A*B*C*D*E*F*G*H*I*J*K*L*M*N*O*P*Q*R*S*T*U*V*W*X*Y)
